# COST (Improved) — checkpoint inference and snapshot ensemble predictions

This is a **post-training** regeneration utility for the single-model and five-snapshot ensemble prediction outputs reported in the paper. Requires access to the original, fitted checkpoints and local cached data. It is not a new training run.


# Regenerate COST (Improved) Test Predictions from Saved Checkpoint

Your Colab session that trained the model has ended, so `best_run_v2` is no longer in memory. But your saved checkpoints (`dsrnn_v2_best.pt`, `dsrnn_v2_ensemble_snapshots.pt`) contain the trained model weights — this notebook reloads them, re-runs preprocessing exactly as before (deterministic, same seed), and re-evaluates on the test set to regenerate the raw per-patient predictions needed for significance testing.

**Output:** `cost_improved_predictions.npz` (same file the significance-testing notebook expects).

In [ ]:
!pip install -q scikit-learn

import os, math
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MIMIC_BASE  = Path('/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1')
CACHE_FILE  = MIMIC_BASE / 'mimic_cache' / 'hf_cohort_processed.npz'
SAVE_DIR    = MIMIC_BASE / 'saved_models'
PRED_DIR    = MIMIC_BASE / 'saved_predictions'
PRED_DIR.mkdir(exist_ok=True)

best_ckpt_path = SAVE_DIR / 'dsrnn_v2_best.pt'
ens_ckpt_path  = SAVE_DIR / 'dsrnn_v2_ensemble_snapshots.pt'

assert CACHE_FILE.exists(), f'Cache not found: {CACHE_FILE} — this is the same cache your training notebook used.'
assert best_ckpt_path.exists(), f'Missing: {best_ckpt_path}'
assert ens_ckpt_path.exists(), f'Missing: {ens_ckpt_path}'
print('All required files found.')

## Feature definitions & config (must match training exactly)

In [ ]:
VITAL_SIGNS = ['heart_rate','sbp','dbp','map','resp_rate','spo2','temperature']
LAB_VALUES  = [
    'creatinine','bnp','lactate','troponin',
    'sodium','potassium','chloride','bicarbonate',
    'bun','glucose','hemoglobin','wbc',
    'platelets','inr','pt','ptt',
    'ph','pao2','paco2','fio2',
    'albumin','bilirubin_total','alt','ast'
]
STATIC_FEATURES = [
    'age','gender_male','admission_emergency','admission_urgent',
    'elixhauser_score','hypertension','diabetes','ckd','copd','af','prior_hf'
]
ALL_TEMPORAL_FEATURES = VITAL_SIGNS + LAB_VALUES
N_FEATURES = len(ALL_TEMPORAL_FEATURES)
N_STATIC   = len(STATIC_FEATURES)
N_INPUT    = N_FEATURES * 3

CONFIG_V2 = {
    'train_years': range(2008, 2016),
    'val_years'  : range(2016, 2018),
    'test_years' : range(2018, 2020),
    'batch_size' : 64,
}
print(f'N_FEATURES={N_FEATURES}  N_STATIC={N_STATIC}  N_INPUT={N_INPUT}')

## Preprocessing (identical to training notebook — deterministic given same SEED + cache)

In [ ]:
def add_missingness_features(X_temporal):
    n, seq, feats = X_temporal.shape
    mask = (~np.isnan(X_temporal)).astype(np.float32)
    delta = np.zeros_like(X_temporal, dtype=np.float32)
    for i in range(n):
        for k in range(feats):
            last_obs = 0
            for t in range(seq):
                if not np.isnan(X_temporal[i, t, k]):
                    last_obs = t
                    delta[i, t, k] = 0.0
                else:
                    delta[i, t, k] = float(t - last_obs)
    delta = delta / (seq + 1e-8)

    values = X_temporal.copy()
    n_vital = len(VITAL_SIGNS)
    for i in range(n):
        for k in range(n_vital):
            last = 0.0
            for t in range(seq):
                if not np.isnan(values[i, t, k]):
                    last = values[i, t, k]
                else:
                    values[i, t, k] = last
    values = np.nan_to_num(values, nan=0.0)

    X_enhanced = np.concatenate([values, mask, delta], axis=-1)
    return X_enhanced.astype(np.float32)


def preprocess_v2(X_temporal, X_static, labels, years, config):
    train_mask = np.isin(years, list(config['train_years']))
    val_mask   = np.isin(years, list(config['val_years']))
    test_mask  = np.isin(years, list(config['test_years']))

    splits_raw = {}
    for name, m in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
        splits_raw[name] = {
            'X_temp': X_temporal[m].copy(),
            'X_stat': X_static[m].copy(),
            'y'     : labels[m].copy()
        }
        print(f'   {name:5s}: {m.sum():,} pasien | pos: {labels[m].sum()} ({labels[m].mean()*100:.1f}%)')

    print('\n⏳ MICE imputation lab values...')
    n_vital = len(VITAL_SIGNS)
    imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)

    tr = splits_raw['train']['X_temp']
    n_tr, seq, n_feat = tr.shape
    lab_train = tr[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
    lab_train = np.nan_to_num(lab_train, nan=0.0, posinf=0.0, neginf=0.0)
    imp.fit(lab_train)

    for name in splits_raw:
        arr = splits_raw[name]['X_temp']
        n_sp = arr.shape[0]
        lab_d = arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital)
        lab_d = np.nan_to_num(lab_d, nan=0.0, posinf=0.0, neginf=0.0)
        arr[:, :, n_vital:] = imp.transform(lab_d).reshape(n_sp, seq, n_feat - n_vital)
        splits_raw[name]['X_temp'] = arr
    print('   ✅ MICE selesai')

    splits = {}
    for name in splits_raw:
        enhanced = add_missingness_features(splits_raw[name]['X_temp'])
        splits[name] = {
            'X_temp': enhanced,
            'X_stat': splits_raw[name]['X_stat'].copy(),
            'y'     : splits_raw[name]['y'].copy()
        }

    tr_flat = splits['train']['X_temp'][:, :, :N_FEATURES].reshape(-1, N_FEATURES)
    feat_mean = tr_flat.mean(axis=0)
    feat_std  = tr_flat.std(axis=0) + 1e-8
    stat_mean = splits['train']['X_stat'].mean(axis=0)
    stat_std  = splits['train']['X_stat'].std(axis=0) + 1e-8

    for name in splits:
        arr = splits[name]['X_temp'].copy()
        arr[:, :, :N_FEATURES] = (arr[:, :, :N_FEATURES] - feat_mean) / feat_std
        splits[name]['X_temp'] = arr
        splits[name]['X_stat'] = (splits[name]['X_stat'] - stat_mean) / stat_std

    return splits


print('⏳ Loading cache & preprocessing (this reproduces the same split/imputation as training)...')
d = np.load(CACHE_FILE, allow_pickle=True)
splits = preprocess_v2(d['X_temporal'], d['X_static'], d['labels'], d['years'], CONFIG_V2)
print('✅ Preprocessing done. Test set shape:', splits['test']['X_temp'].shape)

## Rebuild test DataLoader

In [ ]:
class ICUDataset(Dataset):
    def __init__(self, X_temporal, X_static, y):
        self.X_temp = torch.FloatTensor(X_temporal)
        self.X_stat = torch.FloatTensor(X_static)
        self.y      = torch.FloatTensor(y)
    def __len__(self):          return len(self.y)
    def __getitem__(self, idx): return self.X_temp[idx], self.X_stat[idx], self.y[idx]

test_ds = ICUDataset(splits['test']['X_temp'], splits['test']['X_stat'], splits['test']['y'])
test_loader = DataLoader(test_ds, batch_size=CONFIG_V2['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
print(f'Test set: {len(test_ds)} patients, {len(test_loader)} batches')

## Model architecture (copied verbatim from training notebook)

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=256, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class ClinicalSalienceV2(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_features):
        super().__init__()
        self.fc1  = nn.Linear(input_dim, hidden_dim * 2)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc2  = nn.Linear(hidden_dim, n_features)
        self.tau  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x_t, h_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)
        h = self.fc1(combined)
        h, gate = h.chunk(2, dim=-1)
        h = h * torch.sigmoid(gate)
        h = self.norm(h)
        s_t = torch.sigmoid(self.fc2(h))
        m_hard = (s_t > self.tau).float()
        m_t = m_hard - s_t.detach() + s_t
        return m_t, s_t


class TemporalMultiHeadAttn(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        out, weights = self.attn(x, x, x)
        x = x + out
        x = self.norm(x)
        return x, weights


class StaticTemporalFusion(nn.Module):
    def __init__(self, d_temporal, n_static, d_out, n_heads=4, dropout=0.1):
        super().__init__()
        self.static_proj = nn.Linear(n_static, d_temporal)
        self.cross_attn  = nn.MultiheadAttention(embed_dim=d_temporal, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_temporal)
        self.ffn  = nn.Sequential(
            nn.Linear(d_temporal * 2, d_temporal), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_temporal, d_out)
        )
    def forward(self, static_feat, hidden_states):
        q = self.static_proj(static_feat).unsqueeze(1)
        ctx, attn_w = self.cross_attn(q, hidden_states, hidden_states)
        ctx = self.norm(ctx.squeeze(1))
        pooled = hidden_states.mean(dim=1)
        z = self.ffn(torch.cat([ctx, pooled], dim=-1))
        return z, attn_w.squeeze(1)


class DSRNNv2(nn.Module):
    def __init__(self, n_input, n_static, hidden_size, n_lstm_layers,
                 n_heads, attention_dim, salience_hidden,
                 dropout=0.3, stoch_depth_p=0.1, lambda_sparse=5e-4):
        super().__init__()
        self.hidden_size   = hidden_size
        self.n_lstm_layers = n_lstm_layers
        self.lambda_sparse = lambda_sparse

        self.input_proj = nn.Sequential(
            nn.Linear(n_input, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_enc = SinusoidalPE(hidden_size, dropout=dropout * 0.5)
        self.salience = ClinicalSalienceV2(hidden_size + hidden_size, salience_hidden, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if n_lstm_layers > 1 else 0)
        self.bidir_proj = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size), nn.LayerNorm(hidden_size), nn.GELU()
        )
        self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
        self.fusion = StaticTemporalFusion(hidden_size, n_static, attention_dim, n_heads, dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim), nn.Dropout(dropout),
            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(attention_dim // 2, 1)
        )

    def forward(self, x_temporal, x_static, return_states=False):
        batch, seq, _ = x_temporal.shape
        x_proj = self.input_proj(x_temporal)
        x_proj = self.pos_enc(x_proj)

        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)
        h_prev = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        gated_seq = []
        for t in range(seq):
            x_t = x_proj[:, t, :]
            m_t, s_t = self.salience(x_t, h_prev)
            x_gated = m_t * x_t + (1 - m_t) * x_t * 0.1
            gated_seq.append(x_gated.unsqueeze(1))
            h_prev = h_prev * 0.9 + x_gated.detach() * 0.1
            sparsity_loss = sparsity_loss + m_t.norm(p=1)
        x_gated_seq = torch.cat(gated_seq, dim=1)
        sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)

        lstm_out, _ = self.lstm(x_gated_seq)
        lstm_out = self.bidir_proj(lstm_out)
        H, attn_weights = self.temporal_attn(lstm_out)
        z, cross_attn_w = self.fusion(x_static, H)
        logits = self.classifier(z).squeeze(-1)
        if return_states:
            return logits, sparsity_loss, H, attn_weights
        return logits, sparsity_loss


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x_temp, x_stat, y in loader:
        x_temp, x_stat = x_temp.to(DEVICE), x_stat.to(DEVICE)
        logits, _ = model(x_temp, x_stat)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)

print('✅ Architecture defined')

## Load checkpoints & regenerate predictions

In [ ]:
# --- Load single-best model ---
ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
cfg = ckpt['config']

model = DSRNNv2(
    n_input=N_INPUT, n_static=N_STATIC,
    hidden_size=cfg['hidden_size'], n_lstm_layers=cfg['n_lstm_layers'],
    n_heads=cfg['n_heads'], attention_dim=cfg['attention_dim'],
    salience_hidden=cfg['salience_hidden'], dropout=cfg['dropout'],
    lambda_sparse=cfg['lambda_sparse']
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

test_labels, probs_single = evaluate(model, test_loader)
auc_single = roc_auc_score(test_labels, probs_single)
print(f'Single model  — AUC: {auc_single:.4f}  (checkpoint recorded: {float(ckpt["test_auc_single"]):.4f})')
if abs(auc_single - float(ckpt['test_auc_single'])) > 0.005:
    print('⚠️  AUC differs from checkpoint\'s recorded value by >0.005 — preprocessing may not exactly match training. Investigate before trusting these predictions.')
else:
    print('✅ Matches checkpoint\'s recorded AUC closely — preprocessing reproduced correctly.')

In [ ]:
# --- Load ensemble snapshots, average predictions ---
ens_ckpt = torch.load(ens_ckpt_path, map_location=DEVICE, weights_only=False)
snapshots = ens_ckpt['snapshots']  # list of (val_auc, state_dict)

all_probs_ensemble = []
for val_auc, state in snapshots:
    model.load_state_dict(state)
    _, probs = evaluate(model, test_loader)
    all_probs_ensemble.append(probs)

probs_ensemble = np.mean(all_probs_ensemble, axis=0)
auc_ensemble = roc_auc_score(test_labels, probs_ensemble)
print(f'Ensemble ({len(snapshots)} snapshots) — AUC: {auc_ensemble:.4f}  (checkpoint recorded: {float(ckpt["test_auc_ensemble"]):.4f})')
if abs(auc_ensemble - float(ckpt['test_auc_ensemble'])) > 0.005:
    print('⚠️  AUC differs from checkpoint\'s recorded value by >0.005 — investigate before trusting these predictions.')
else:
    print('✅ Matches checkpoint\'s recorded AUC closely.')

## Save — this is the file the significance-testing notebook needs

In [ ]:
np.savez(
    PRED_DIR / 'cost_improved_predictions.npz',
    test_labels    = test_labels,
    probs_single   = probs_single,
    probs_ensemble = probs_ensemble,
)
print(f'✅ Saved: {PRED_DIR / "cost_improved_predictions.npz"}')
print('\nYou can now go back to the significance-testing notebook and re-run it — ',
      'the COST Improved half will load successfully.')
print('\n⚠️  Baseline predictions (LSTM, Attention LSTM, COST Before) are still needed ',
      'separately — this notebook only regenerates COST Improved.')